# Продукт дашборда: каталог и условия поездки

Что показываем в дашборде: сценарии стартового каталога и условия поездки
(сезонность + прогноз). Секции: стартовый каталог → условия поездки.

## Стартовый каталог (Этап 7)

Управленческие сценарии подбора стартового каталога: для N = 5/8/10/12 и трёх
стратегий (топ / региональное разнообразие / разнообразие + подтверждённые
координаты) показываем компромисс между средним индексом и разнообразием.

Скрипт: `src/analytics/catalog_builder.py`.
Витрины: `data/mart/catalog_scenarios.csv`, `catalog_members.csv`.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root))
PROJECT_ROOT = _root
from src.analytics.catalog_builder import build_scenarios, load_scores

scores = load_scores()
scenarios, members = build_scenarios(scores)

scen = pd.DataFrame(scenarios)
mem = pd.DataFrame(members)
print("сценариев:", len(scen), "| записей состава:", len(mem))


### 1. Стратегии подбора

- **top** — топ-N по индексу (без региональных ограничений);
- **diversity** — не более одного курорта на регион (максимум разнообразия);
- **diversity_verified** — то же, но при прочих равных предпочитаются курорты с
  подтверждёнными координатами (`coordinate_validation_status == "confirmed"`).

Целевая функция — сумма базовых индексов, без оптимизации прибыли.


In [ ]:
# Скелет: стратегия «региональное разнообразие» — не более 1 курорта на регион
best = {}
for r in scores:
    cur = best.get(r["region"])
    if cur is None or r["popularity_score"] > cur["popularity_score"]:
        best[r["region"]] = r

best_list = sorted(best.values(), key=lambda r: (-r["popularity_score"], r["resort_id"]))
print(f"Лучший курорт каждого из {len(best)} регионов; топ-8 по индексу:")
print(pd.DataFrame(best_list[:8])[["name_ru", "region", "popularity_score"]].to_string(index=False))


### 2. Сравнение с топ-N

Для каждого сценария считаем KPI: число регионов, концентрация (максимальная
доля одного региона), средний индекс и разница с топ-N, покрытие ценами и
подтверждёнными координатами.


In [ ]:
cols = ["scenario_id", "n", "region_count", "max_region_share",
        "avg_index", "avg_index_vs_top", "observed_price_count", "confirmed_coords_count"]
show = scen[cols].copy()
print(show.to_string(index=False))


### 3. Состав одного сценария (пример)

Смотрим, что реально попадёт в каталог и чем отличается от топ-N.


In [ ]:
scid = "diversity_n8"
sel = mem[mem["scenario_id"] == scid].sort_values("popularity_score", ascending=False)
print(f"Состав {scid}:")
print(sel[["name_ru", "region", "popularity_score", "rank"]].to_string(index=False))


In [ ]:
# Замены относительно топ-8
top = mem[mem["scenario_id"] == "top_n8"]
div = mem[mem["scenario_id"] == "diversity_n8"]
added = set(div["resort_id"]) - set(top["resort_id"])
removed = set(top["resort_id"]) - set(div["resort_id"])
print("Добавлены (нет в топ-8):", div[div["resort_id"].isin(added)]["name_ru"].tolist())
print("Исключены (были в топ-8):", top[top["resort_id"].isin(removed)]["name_ru"].tolist())


### Итог

Разнообразие — это компромисс: топ-N концентрируется в нескольких регионах
(высокая концентрация), а `diversity` расширяет региональное покрытие ценой
нескольких баллов среднего индекса.


## Условия поездки (Этап 8)

Разворачиваем семидневный прогноз в плоскую таблицу «курорт × дата», смягчаем
категории состояния трассы («предпосылки…», а не категоричный «лёд»/«накат») и
добавляем причину и ограничение оценки.

Скрипт: `src/analytics/trip_conditions.py`.
Витрина: `data/mart/weather_forecast_daily.csv`.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root))
PROJECT_ROOT = _root
from src.analytics.trip_conditions import build_daily_rows, load_forecast

payload = load_forecast()
rows = build_daily_rows(payload)
df = pd.DataFrame(rows)
print("строк (курорт × дата):", len(df), "| курортов:", df["resort_id"].nunique())


### 1. Мягкие категории

Категории — погодные предпосылки, а не наблюдаемая поверхность:
«предпосылки свежего рыхлого снега», «предпосылки увлажнения/размягчения»,
«предпосылки замерзания» (только при достаточных данных), «модель не показывает
естественный снежный покров», «оценки недостаточно». Дневные min/max не
фиксируют последовательность оттепели и замерзания.


In [ ]:
# Пороги правила классификации (та же функция, что и в fetch_weather_forecast.py)
from src.collect.fetch_weather_forecast import classify_slope_explained

cases = [
    # (снегопад см, дождь мм, t_max, t_min, высота снега м)
    (15.0, 0.0, -5.0, -9.0, 0.5),  # свежий снег и мороз
    (0.0, 3.0, 3.0, 0.0, 0.4),      # дождь
    (0.0, 0.0, 1.0, -5.0, 0.2),      # суточный ход температуры через 0
    (0.0, 0.0, 5.0, 1.0, 0.0),       # нет покрова и снегопада
    (0.0, 0.0, -4.0, -8.0, 0.6),      # мороз, но без признаков свежего снега
]
for c in cases:
    cond, reason, limits = classify_slope_explained(*c)
    print(f"{c} -> {cond}")
    print(f"    причина: {reason}")
    print(f"    ограничение: {limits}")


In [ ]:
print("Распределение категорий:")
print(df["slope_condition"].value_counts().to_string())
print()
print("Пакет прогноза:", payload.get("fetched_at"), "| горизонт:", payload.get("forecast_days"), "дней")


### 2. Разворот прогноза (курорт × дата)

Одна строка = курорт × дата прогноза. Свежесть (текущее время − fetched_at)
считается на стороне BI, а не зашивается в статичный JSON.


In [ ]:
rid = "R004"  # Шерегеш
r = df[df["resort_id"] == rid].sort_values("date")
cols = ["date", "temperature_min_c", "temperature_max_c", "snowfall_cm",
        "rain_mm", "snow_depth_m", "slope_condition"]
print(r[cols].to_string(index=False))


### 3. Причины и ограничения

Для каждого дня хранится `condition_reason` (почему) и
`condition_limitations` (чего оценка не учитывает).


In [ ]:
row = df.iloc[0]
print(f"{row['resort_id']} {row['name_ru']} {row['date']}: {row['slope_condition']}")
print("  причина:", row["condition_reason"])
print("  ограничение:", row["condition_limitations"])


### Итог

Прогноз на 7 дней по 32 курортам = 224 строки. Распределение категорий в
середине октября закономерно смещено к «модель не показывает естественный
снежный покров» (межсезонье).
